# convtranspose-bn-activation-block — worked example 2: Inner block with LeakyReLU activation

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `convtranspose-bn-activation-block`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Some generator variants swap plain `ReLU` for `LeakyReLU(0.2)` so a small negative gradient flows even when the pre-activation is negative, reducing dead units. The structure is otherwise the canonical `ConvTranspose2d(bias=False) -> BatchNorm2d -> activation`, still doubling spatial size with `kernel=4, stride=2, padding=1`.

## Worked solution

**Step 1 — Same skeleton as the canonical block.** It is still three layers in order: ConvTranspose, then BatchNorm, then an activation. Only the activation changes.

**Step 2 — bias=False on the ConvTranspose.** BatchNorm immediately follows and its learnable `beta` subsumes any conv bias, so a conv bias would be wasted parameters that BN cancels. Set `bias=False`.

**Step 3 — Use LeakyReLU(0.2).** `nn.LeakyReLU(0.2, inplace=True)` multiplies negative inputs by 0.2 instead of zeroing them. The slope 0.2 is the DCGAN convention. `inplace=True` saves memory by overwriting the BN output buffer.

**Step 4 — Confirm doubling and the leak.** With `kernel=4, stride=2, padding=1`, an 8x8 input becomes 16x16. To sanity-check the activation behaves like a leak, feed a negative value through a bare `LeakyReLU(0.2)`: -10 maps to -2.0 (i.e. 0.2 * -10), not 0.

In [ ]:
import torch.nn as nn

def build_leaky_block(in_channels, out_channels):
    return nn.Sequential(
        nn.ConvTranspose2d(in_channels, out_channels, kernel_size=4, stride=2, padding=1, bias=False),
        nn.BatchNorm2d(out_channels),
        nn.LeakyReLU(0.2, inplace=True),
    )

t.manual_seed(0)
block = build_leaky_block(128, 64)
x = t.randn(2, 128, 8, 8)
out = block(x)
print('output shape:', tuple(out.shape))
leak = nn.LeakyReLU(0.2)
print('leak of -10:', leak(t.tensor(-10.0)).item())
print('convt bias is None:', block[0].bias is None)